# Multiple Linear Regression Assignment Solutions
## Predicting Urban Energy Consumption - NordicEnergy Analytics
### TTK4260 - Multivariate Data Analysis and Machine Learning

---

This notebook provides detailed solutions with explanations for all questions in the MLR assignment.

In [ ]:
# Install required packages
!pip install numpy pandas plotly scikit-learn scipy statsmodels -q

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("All libraries imported successfully!")

---
# Question 1: Multiple Regression Fundamentals
## Building Energy Model with 8 Buildings

We start with a basic multiple regression problem: predicting building energy consumption from Area, Age, and number of Floors.

In [ ]:
# Question 1 Data
q1_data = pd.DataFrame({
    'Building': [1, 2, 3, 4, 5, 6, 7, 8],
    'Area': [1200, 2500, 800, 3200, 1800, 950, 2100, 1500],
    'Age': [5, 15, 8, 25, 12, 3, 20, 10],
    'Floors': [3, 5, 2, 8, 4, 2, 6, 4],
    'Energy': [285, 520, 195, 780, 390, 210, 545, 340]
})

print("Question 1 Dataset:")
q1_data

## Part (a): Design Matrix and Response Vector

The multiple linear regression model is:
$$y = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \theta_3 x_3 + \varepsilon$$

In matrix form: $\mathbf{y} = \mathbf{X}\boldsymbol{\theta} + \boldsymbol{\varepsilon}$

The **design matrix** $\mathbf{X}$ includes:
- Column of 1s for the intercept ($\theta_0$)
- Columns for each predictor variable

In [ ]:
# Part (a): Set up design matrix X and response vector y

# Response vector
y = q1_data['Energy'].values

# Design matrix with intercept column
n = len(q1_data)
X = np.column_stack([
    np.ones(n),           # Intercept column
    q1_data['Area'],      # x1
    q1_data['Age'],       # x2
    q1_data['Floors']     # x3
])

print("Design Matrix X dimensions:", X.shape)
print(f"  - {X.shape[0]} observations (rows)")
print(f"  - {X.shape[1]} parameters (columns: intercept + 3 features)")
print("\nResponse Vector y dimensions:", y.shape)
print(f"  - {y.shape[0]} observations")

print("\n" + "="*60)
print("First 3 rows of Design Matrix X:")
print("="*60)
print("     [Intercept]  [Area]    [Age]    [Floors]")
for i in range(3):
    print(f"Row {i+1}:  {X[i,0]:.0f}        {X[i,1]:.0f}      {X[i,2]:.0f}        {X[i,3]:.0f}")

print("\nFirst 3 elements of y:", y[:3])

## Part (b): Computing $\mathbf{X}^T\mathbf{X}$ and $\mathbf{X}^T\mathbf{y}$

These are the key components of the **normal equations**:
- $\mathbf{X}^T\mathbf{X}$ is a $(p+1) \times (p+1)$ matrix (4×4 in our case)
- $\mathbf{X}^T\mathbf{y}$ is a $(p+1) \times 1$ vector

In [ ]:
# Part (b): Compute X^T X and X^T y

XtX = X.T @ X  # Matrix multiplication
Xty = X.T @ y

print("X^T X (4×4 matrix):")
print(f"Dimensions: {XtX.shape}")
print("\nMatrix:")
print(np.round(XtX, 2))

print("\n" + "="*60)
print("\nX^T y (4×1 vector):")
print(f"Dimensions: {Xty.shape}")
print("\nVector:")
print(np.round(Xty, 2))

# Interpretation of X^T X diagonal elements
print("\n" + "="*60)
print("\nInterpretation of X^T X diagonal:")
print(f"  - (1,1): Sum of 1s = n = {XtX[0,0]:.0f}")
print(f"  - (2,2): Sum of Area² = {XtX[1,1]:.0f}")
print(f"  - (3,3): Sum of Age² = {XtX[2,2]:.0f}")
print(f"  - (4,4): Sum of Floors² = {XtX[3,3]:.0f}")

## Part (c): Solving the Normal Equations

The OLS solution is:
$$\hat{\boldsymbol{\theta}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

In [ ]:
# Part (c): Solve normal equations

# Method 1: Direct inverse (educational, but less stable numerically)
theta_hat = np.linalg.inv(XtX) @ Xty

# Method 2: Using np.linalg.solve (more numerically stable)
theta_hat_stable = np.linalg.solve(XtX, Xty)

print("OLS Coefficient Estimates:")
print("="*60)
print(f"θ₀ (Intercept) = {theta_hat[0]:.4f}")
print(f"θ₁ (Area)      = {theta_hat[1]:.6f}")
print(f"θ₂ (Age)       = {theta_hat[2]:.4f}")
print(f"θ₃ (Floors)    = {theta_hat[3]:.4f}")

# Verify with sklearn
print("\n" + "="*60)
print("Verification with sklearn LinearRegression:")
lr = LinearRegression()
lr.fit(q1_data[['Area', 'Age', 'Floors']], q1_data['Energy'])
print(f"Intercept: {lr.intercept_:.4f}")
print(f"Coefficients: {lr.coef_}")

## Part (d): Coefficient Interpretation - "Holding Others Fixed"

This is the **ceteris paribus** (all else equal) interpretation, which is fundamental to multiple regression.

In [ ]:
# Part (d): Interpretation of coefficients

print("COEFFICIENT INTERPRETATION")
print("="*70)

print("\n📊 θ₀ (Intercept) = {:.2f} MWh/year".format(theta_hat[0]))
print("-"*70)
print("   This is the predicted energy consumption for a hypothetical building")
print("   with Area=0, Age=0, and Floors=0.")
print("   ⚠️  This is often not meaningful - just a mathematical baseline.")

print("\n📊 θ₁ (Area) = {:.6f} MWh/m²/year".format(theta_hat[1]))
print("-"*70)
print("   Interpretation: For each additional 1 m² of building area,")
print("   energy consumption increases by {:.4f} MWh/year,".format(theta_hat[1]))
print("   HOLDING AGE AND FLOORS CONSTANT.")
print("   ")
print("   Practical: A 100 m² increase → +{:.2f} MWh/year".format(theta_hat[1]*100))

print("\n📊 θ₂ (Age) = {:.4f} MWh/year per year of age".format(theta_hat[2]))
print("-"*70)
print("   Interpretation: For each additional year of building age,")
print("   energy consumption increases by {:.2f} MWh/year,".format(theta_hat[2]))
print("   HOLDING AREA AND FLOORS CONSTANT.")
print("   ")
print("   This suggests older buildings are less energy efficient,")
print("   likely due to degraded insulation, older HVAC systems, etc.")

print("\n📊 θ₃ (Floors) = {:.4f} MWh/floor/year".format(theta_hat[3]))
print("-"*70)
print("   Interpretation: Each additional floor adds {:.2f} MWh/year,".format(theta_hat[3]))
print("   HOLDING AREA AND AGE CONSTANT.")
print("   ")
print("   Note: This might capture vertical heat loss, elevator energy,")
print("   or other floor-specific energy demands.")

print("\n" + "="*70)
print("⚠️  KEY INSIGHT: The 'holding others fixed' interpretation is CRITICAL!")
print("   Without it, we cannot separate the individual effects of each variable.")
print("="*70)

## Part (e): Model Fit - R² and Residuals

In [ ]:
# Part (e): Calculate R² and residuals

# Predictions
y_pred = X @ theta_hat

# Residuals
residuals = y - y_pred

# R-squared calculation
SS_tot = np.sum((y - np.mean(y))**2)  # Total sum of squares
SS_res = np.sum(residuals**2)          # Residual sum of squares
R_squared = 1 - SS_res / SS_tot

print("MODEL FIT STATISTICS")
print("="*60)
print(f"R² = {R_squared:.6f}")
print(f"This means {R_squared*100:.2f}% of the variance in Energy is")
print(f"explained by Area, Age, and Floors.")

print("\n" + "="*60)
print("RESIDUAL ANALYSIS")
print("="*60)

results_df = pd.DataFrame({
    'Building': q1_data['Building'],
    'Actual (y)': y,
    'Predicted (ŷ)': np.round(y_pred, 2),
    'Residual (e)': np.round(residuals, 2),
    '|Residual|': np.round(np.abs(residuals), 2)
})
print(results_df.to_string(index=False))

# Find largest error
max_error_idx = np.argmax(np.abs(residuals))
print("\n" + "="*60)
print(f"🔍 LARGEST PREDICTION ERROR: Building {max_error_idx + 1}")
print(f"   Residual = {residuals[max_error_idx]:.2f} MWh/year")
print(f"   Actual = {y[max_error_idx]} MWh, Predicted = {y_pred[max_error_idx]:.1f} MWh")
print("\n   Possible explanations:")
print("   - Unusual building usage patterns")
print("   - Missing important features (e.g., insulation quality)")
print("   - Data collection error")
print("   - Non-linear relationships not captured by the model")

In [ ]:
# Visualization of residuals
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=('Actual vs Predicted', 'Residuals by Building'))

# Actual vs Predicted
fig.add_trace(
    go.Scatter(x=y_pred, y=y, mode='markers', 
               marker=dict(size=12, color='blue'),
               text=[f'Building {i+1}' for i in range(len(y))],
               name='Buildings'),
    row=1, col=1
)
# Perfect prediction line
fig.add_trace(
    go.Scatter(x=[min(y_pred), max(y_pred)], y=[min(y_pred), max(y_pred)],
               mode='lines', line=dict(dash='dash', color='red'),
               name='Perfect Fit'),
    row=1, col=1
)

# Residual bar chart
colors = ['green' if r > 0 else 'red' for r in residuals]
fig.add_trace(
    go.Bar(x=[f'B{i+1}' for i in range(len(residuals))], y=residuals,
           marker_color=colors, name='Residuals'),
    row=1, col=2
)

fig.update_layout(height=400, width=900, title_text="Question 1(e): Model Fit Analysis",
                  showlegend=False)
fig.update_xaxes(title_text="Predicted Energy (MWh)", row=1, col=1)
fig.update_yaxes(title_text="Actual Energy (MWh)", row=1, col=1)
fig.update_xaxes(title_text="Building", row=1, col=2)
fig.update_yaxes(title_text="Residual (MWh)", row=1, col=2)
fig.show()

---
# Question 2: The Importance of Including Confounders

This question explores **omitted variable bias** - what happens when we leave out an important variable that is correlated with both our predictor and outcome.

In [ ]:
# Part (a): Simple regression with Area only

X_simple = np.column_stack([np.ones(n), q1_data['Area']])
theta_simple = np.linalg.solve(X_simple.T @ X_simple, X_simple.T @ y)

print("SIMPLE REGRESSION MODEL: Energy ~ Area")
print("="*60)
print(f"θ₀ (Intercept) = {theta_simple[0]:.4f}")
print(f"θ₁ (Area)      = {theta_simple[1]:.6f} MWh/m²")
print(f"\nθ₁_simple = {theta_simple[1]:.6f}")

In [ ]:
# Part (b): Compare simple vs multiple regression coefficients

print("COEFFICIENT COMPARISON")
print("="*60)
print(f"Simple Regression (Area only):")
print(f"   θ₁_simple = {theta_simple[1]:.6f} MWh/m²")
print(f"\nMultiple Regression (Area + Age + Floors):")
print(f"   θ₁_multiple = {theta_hat[1]:.6f} MWh/m²")

print("\n" + "="*60)
difference = theta_simple[1] - theta_hat[1]
pct_diff = (difference / theta_hat[1]) * 100
print(f"DIFFERENCE: {difference:.6f} MWh/m²")
print(f"PERCENTAGE: {pct_diff:.1f}% higher in simple regression")

print("\n🔍 OBSERVATION:")
if theta_simple[1] > theta_hat[1]:
    print("   The simple regression OVERESTIMATES the effect of Area!")
else:
    print("   The simple regression UNDERESTIMATES the effect of Area!")

## Part (c): Omitted Variable Bias Explanation

### Causal Diagram

```
         Age
        ↙   ↘
    Area  →  Energy
```

- **Age → Area**: Older buildings tend to be larger (built when land was cheaper)
- **Age → Energy**: Older buildings are less efficient (worse insulation, older systems)
- **Area → Energy**: Larger buildings use more energy

When we omit Age, the coefficient for Area "absorbs" part of Age's effect!

In [ ]:
# Part (c): Visualize the omitted variable bias

print("OMITTED VARIABLE BIAS EXPLANATION")
print("="*70)
print("""
When we fit: Energy ~ Area (omitting Age)

The estimated coefficient θ₁_simple captures TWO effects:

1. DIRECT EFFECT: Area → Energy
   (True effect of area on energy consumption)

2. INDIRECT EFFECT: Area ← Age → Energy  
   (Effect of Age that 'travels through' the correlation with Area)

MATHEMATICAL FORMULA:
θ₁_simple ≈ θ₁_true + θ₂ × (correlation between Area and Age)

Since:
- θ₂ (Age effect) is POSITIVE (older buildings use more energy)
- Correlation(Area, Age) is POSITIVE (older buildings are larger)

The bias = θ₂ × corr(Area, Age) is POSITIVE
Therefore: θ₁_simple > θ₁_true (OVERESTIMATE)
""")

# Create causal diagram visualization
fig = go.Figure()

# Nodes
fig.add_trace(go.Scatter(
    x=[1, 2, 2], y=[2, 3, 1],
    mode='markers+text',
    marker=dict(size=50, color=['lightblue', 'lightgreen', 'salmon']),
    text=['Area', 'Age', 'Energy'],
    textposition='middle center',
    textfont=dict(size=14, color='black')
))

# Arrows
# Age -> Area
fig.add_annotation(x=1.15, y=2.1, ax=1.85, ay=2.85, xref='x', yref='y',
                   axref='x', ayref='y', showarrow=True,
                   arrowhead=2, arrowsize=1.5, arrowcolor='blue')
# Age -> Energy
fig.add_annotation(x=2, y=1.3, ax=2, ay=2.7, xref='x', yref='y',
                   axref='x', ayref='y', showarrow=True,
                   arrowhead=2, arrowsize=1.5, arrowcolor='blue')
# Area -> Energy
fig.add_annotation(x=1.85, y=1.1, ax=1.15, ay=1.9, xref='x', yref='y',
                   axref='x', ayref='y', showarrow=True,
                   arrowhead=2, arrowsize=1.5, arrowcolor='green')

fig.update_layout(
    title='Causal Diagram: Age as a Confounder',
    xaxis=dict(range=[0.5, 2.5], showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(range=[0.5, 3.5], showgrid=False, zeroline=False, showticklabels=False),
    height=400, width=500
)
fig.show()

In [ ]:
# Part (d): Calculate correlation between Area and Age

corr_area_age = np.corrcoef(q1_data['Area'], q1_data['Age'])[0, 1]

print("CORRELATION ANALYSIS")
print("="*60)
print(f"Correlation(Area, Age) = {corr_area_age:.4f}")

print("\n🔍 INTERPRETATION:")
if corr_area_age > 0.5:
    print(f"   Strong positive correlation ({corr_area_age:.2f})")
elif corr_area_age > 0.3:
    print(f"   Moderate positive correlation ({corr_area_age:.2f})")
else:
    print(f"   Weak positive correlation ({corr_area_age:.2f})")

print("\n   This supports our explanation: older buildings tend to be larger.")
print("   Since Age also affects Energy positively, omitting Age causes")
print("   the Area coefficient to be biased upward.")

# Visualize
fig = px.scatter(q1_data, x='Area', y='Age', 
                 title=f'Area vs Age Correlation (r = {corr_area_age:.3f})',
                 trendline='ols')
fig.update_layout(height=400, width=500)
fig.show()

In [ ]:
# Part (e): Prediction vs Interpretation

print("PREDICTION VS INTERPRETATION: DO WE NEED CONFOUNDERS?")
print("="*70)

# Compare R² of simple vs multiple model
y_pred_simple = X_simple @ theta_simple
R2_simple = 1 - np.sum((y - y_pred_simple)**2) / SS_tot

print(f"R² (Simple: Area only):        {R2_simple:.4f}")
print(f"R² (Multiple: Area+Age+Floors): {R_squared:.4f}")

print("\n" + "="*70)
print("""
WHEN THE COLLEAGUE IS CORRECT:
✓ If the goal is ONLY prediction on data with the SAME correlation structure
✓ If we don't care about understanding WHY variables matter
✓ If the relationship between confounders and predictors remains stable

WHEN THE COLLEAGUE IS WRONG:
✗ If we want to understand the TRUE causal effect of Area
✗ If we need to make predictions under DIFFERENT conditions
  (e.g., new buildings where Age-Area correlation doesn't hold)
✗ If we want to make policy decisions (e.g., "should we prioritize
  renovating old buildings or reducing floor space?")
✗ If the confounding structure changes in future data

EXAMPLE OF FAILURE:
Suppose new regulations require all new large buildings to be energy-efficient.
Now the Area-Energy relationship changes for NEW buildings, but the
Age-Energy relationship stays the same.
→ A model including Age will generalize better!
""")

---
# Question 3: Multicollinearity and Coefficient Stability

When predictors are highly correlated, OLS estimates become **unstable** - small changes in data lead to large changes in coefficients.

In [ ]:
# Question 3 Data
q3_data = pd.DataFrame({
    'ID': range(1, 11),
    'Area': [1000, 1500, 2000, 1200, 2500, 1800, 900, 2200, 1600, 1100],
    'Windows': [180, 275, 360, 220, 455, 330, 165, 400, 290, 200],
    'Age': [10, 8, 15, 5, 20, 12, 7, 18, 11, 6],
    'Energy': [245, 358, 485, 278, 615, 425, 218, 538, 382, 262]
})

print("Question 3 Dataset:")
q3_data

In [ ]:
# Part (a): Correlation between Area and Windows

corr_area_windows = np.corrcoef(q3_data['Area'], q3_data['Windows'])[0, 1]

print("MULTICOLLINEARITY DETECTION")
print("="*60)
print(f"Correlation(Area, Windows) = {corr_area_windows:.4f}")

print("\n🚨 WARNING:")
if corr_area_windows > 0.9:
    print(f"   SEVERE multicollinearity detected (r = {corr_area_windows:.3f})!")
    print("   Including both variables will cause:")
    print("   - Unstable coefficient estimates")
    print("   - High standard errors")
    print("   - Difficulty interpreting individual effects")

# Visualize the correlation
fig = px.scatter(q3_data, x='Area', y='Windows',
                 title=f'Area vs Windows: Nearly Perfect Correlation (r = {corr_area_windows:.4f})',
                 trendline='ols')
fig.update_layout(height=400, width=500)
fig.show()

In [ ]:
# Part (b): Fit three models

y3 = q3_data['Energy'].values

# Model A: Area + Age
X_A = sm.add_constant(q3_data[['Area', 'Age']])
model_A = sm.OLS(y3, X_A).fit()

# Model B: Windows + Age
X_B = sm.add_constant(q3_data[['Windows', 'Age']])
model_B = sm.OLS(y3, X_B).fit()

# Model C: Area + Windows + Age
X_C = sm.add_constant(q3_data[['Area', 'Windows', 'Age']])
model_C = sm.OLS(y3, X_C).fit()

print("THREE MODEL COMPARISON")
print("="*70)

print("\nModel A: Energy ~ Area + Age")
print("-"*40)
print(f"  Intercept: {model_A.params[0]:.4f}")
print(f"  Area:      {model_A.params[1]:.6f}")
print(f"  Age:       {model_A.params[2]:.4f}")

print("\nModel B: Energy ~ Windows + Age")
print("-"*40)
print(f"  Intercept: {model_B.params[0]:.4f}")
print(f"  Windows:   {model_B.params[1]:.6f}")
print(f"  Age:       {model_B.params[2]:.4f}")

print("\nModel C: Energy ~ Area + Windows + Age")
print("-"*40)
print(f"  Intercept: {model_C.params[0]:.4f}")
print(f"  Area:      {model_C.params[1]:.6f}")
print(f"  Windows:   {model_C.params[2]:.6f}")
print(f"  Age:       {model_C.params[3]:.4f}")

print("\n🔍 NOTICE how Area and Windows coefficients in Model C")
print("   are VERY different from Models A and B!")

In [ ]:
# Part (c): Compare R²

print("R² COMPARISON")
print("="*60)
print(f"Model A (Area + Age):           R² = {model_A.rsquared:.6f}")
print(f"Model B (Windows + Age):        R² = {model_B.rsquared:.6f}")
print(f"Model C (Area + Windows + Age): R² = {model_C.rsquared:.6f}")

print("\n" + "="*60)
improvement = model_C.rsquared - model_A.rsquared
print(f"Improvement from adding Windows to Model A: {improvement:.6f}")
print(f"Percentage improvement: {improvement/model_A.rsquared*100:.2f}%")

print("\n🔍 CONCLUSION:")
print("   Adding Windows to a model that already has Area provides")
print("   ALMOST NO improvement in R²!")
print("   This is because Area and Windows contain nearly the SAME information.")
print("   They are essentially measuring the same underlying concept: building size.")

In [ ]:
# Part (d): Bootstrap analysis of coefficient stability

np.random.seed(42)
n_bootstrap = 500
n_samples = len(q3_data)

# Store bootstrap coefficients
boot_area = []
boot_windows = []

for _ in range(n_bootstrap):
    # Resample with replacement
    idx = np.random.choice(n_samples, size=n_samples, replace=True)
    X_boot = X_C.iloc[idx]
    y_boot = y3[idx]
    
    # Fit model
    try:
        model_boot = sm.OLS(y_boot, X_boot).fit()
        boot_area.append(model_boot.params['Area'])
        boot_windows.append(model_boot.params['Windows'])
    except:
        pass  # Skip if singular matrix

boot_area = np.array(boot_area)
boot_windows = np.array(boot_windows)

print("BOOTSTRAP COEFFICIENT ANALYSIS (500 resamples)")
print("="*60)
print(f"Area coefficient:")
print(f"  Mean:   {np.mean(boot_area):.4f}")
print(f"  Std:    {np.std(boot_area):.4f}")
print(f"  Range:  [{np.min(boot_area):.4f}, {np.max(boot_area):.4f}]")

print(f"\nWindows coefficient:")
print(f"  Mean:   {np.mean(boot_windows):.4f}")
print(f"  Std:    {np.std(boot_windows):.4f}")
print(f"  Range:  [{np.min(boot_windows):.4f}, {np.max(boot_windows):.4f}]")

print("\n🚨 HUGE VARIANCE in both coefficients!")
print("   The estimates are extremely UNSTABLE due to multicollinearity.")

In [ ]:
# Visualize bootstrap distributions
fig = make_subplots(rows=1, cols=2, subplot_titles=('Bootstrap: Area Coefficient', 
                                                      'Bootstrap: Windows Coefficient'))

fig.add_trace(
    go.Histogram(x=boot_area, nbinsx=30, marker_color='steelblue', name='Area'),
    row=1, col=1
)
fig.add_vline(x=model_C.params['Area'], line_dash='dash', line_color='red', row=1, col=1)

fig.add_trace(
    go.Histogram(x=boot_windows, nbinsx=30, marker_color='coral', name='Windows'),
    row=1, col=2
)
fig.add_vline(x=model_C.params['Windows'], line_dash='dash', line_color='red', row=1, col=2)

fig.update_layout(height=400, width=900, 
                  title_text="Bootstrap Coefficient Distributions (High Variance = Unstable)",
                  showlegend=False)
fig.show()

In [ ]:
# Part (e): Variance Inflation Factor (VIF)

def calculate_vif(X, feature_names):
    """Calculate VIF for each feature"""
    vif_data = []
    X_array = X.values if hasattr(X, 'values') else X
    
    for i, name in enumerate(feature_names):
        # Get the feature
        y_vif = X_array[:, i]
        # Get other features
        X_vif = np.delete(X_array, i, axis=1)
        X_vif = sm.add_constant(X_vif)
        
        # Regress feature on others
        model_vif = sm.OLS(y_vif, X_vif).fit()
        r_squared = model_vif.rsquared
        
        # VIF = 1 / (1 - R²)
        vif = 1 / (1 - r_squared) if r_squared < 1 else np.inf
        vif_data.append({'Feature': name, 'R²': r_squared, 'VIF': vif})
    
    return pd.DataFrame(vif_data)

# Calculate VIF for Model C features
X_features = q3_data[['Area', 'Windows', 'Age']]
vif_results = calculate_vif(X_features.values, ['Area', 'Windows', 'Age'])

print("VARIANCE INFLATION FACTOR (VIF) ANALYSIS")
print("="*60)
print(vif_results.to_string(index=False))

print("\n" + "="*60)
print("VIF INTERPRETATION GUIDE:")
print("  VIF = 1:      No multicollinearity")
print("  VIF < 5:      Low multicollinearity")
print("  VIF 5-10:     Moderate multicollinearity")
print("  VIF > 10:     HIGH multicollinearity ⚠️")

print("\n🚨 CONCLUSION:")
print(f"   Area VIF = {vif_results.loc[0, 'VIF']:.1f} - SEVERE multicollinearity!")
print(f"   Windows VIF = {vif_results.loc[1, 'VIF']:.1f} - SEVERE multicollinearity!")
print("   These two variables should NOT both be in the model.")

In [ ]:
# Part (f): The "Who Gets Credit" Problem - visualizing coefficient trade-off

# Many combinations of (θ_area, θ_windows) give similar predictions
# because Area ≈ k * Windows

# The constraint: θ_area * mean(Area) + θ_windows * mean(Windows) ≈ constant
mean_area = q3_data['Area'].mean()
mean_windows = q3_data['Windows'].mean()

# From Model A, the combined effect at means
target_effect = model_A.params['Area'] * mean_area  # This is what we're trying to explain

# Generate line of equivalent solutions
theta_area_range = np.linspace(-0.5, 0.5, 100)
theta_windows_equiv = (target_effect - theta_area_range * mean_area) / mean_windows

fig = go.Figure()

# Line of equivalent solutions
fig.add_trace(go.Scatter(
    x=theta_area_range, y=theta_windows_equiv,
    mode='lines', name='Equivalent Solutions',
    line=dict(color='blue', width=2)
))

# Actual OLS solution
fig.add_trace(go.Scatter(
    x=[model_C.params['Area']], y=[model_C.params['Windows']],
    mode='markers', name='OLS Solution',
    marker=dict(size=15, color='red', symbol='star')
))

# Bootstrap solutions
fig.add_trace(go.Scatter(
    x=boot_area, y=boot_windows,
    mode='markers', name='Bootstrap Samples',
    marker=dict(size=5, color='gray', opacity=0.3)
))

fig.update_layout(
    title="The 'Who Gets Credit' Problem: Many Equivalent Solutions",
    xaxis_title='θ_Area',
    yaxis_title='θ_Windows',
    height=500, width=600
)
fig.show()

print("\n" + "="*70)
print("THE 'WHO GETS CREDIT' PROBLEM")
print("="*70)
print("""
When two predictors are highly correlated, many different combinations
of their coefficients produce nearly identical predictions.

Think of it like paying for a $100 item with two credit cards:
- Card A: $80, Card B: $20  → Total: $100 ✓
- Card A: $50, Card B: $50  → Total: $100 ✓
- Card A: $20, Card B: $80  → Total: $100 ✓

OLS picks ONE solution (the one that minimizes sum of squared residuals),
but this choice is arbitrary and unstable!

The blue line shows all (θ_Area, θ_Windows) combinations that give
approximately the same prediction. Bootstrap samples scatter along this line.
""")

---
# Question 4: Categorical Variables and Dummy Encoding

Categorical variables require special handling - we convert them to **dummy variables** (0/1 indicators).

In [ ]:
# Question 4 Data
q4_data = pd.DataFrame({
    'ID': range(1, 13),
    'Area': [1500, 1200, 2000, 1800, 1000, 2500, 1600, 1400, 2200, 1100, 1300, 1900],
    'Type': ['Office', 'Retail', 'Industrial', 'Office', 'Retail', 'Industrial',
             'Office', 'Retail', 'Industrial', 'Office', 'Retail', 'Industrial'],
    'Age': [10, 5, 15, 8, 3, 20, 12, 7, 18, 6, 4, 12],
    'Energy': [380, 420, 520, 440, 355, 640, 395, 485, 570, 285, 450, 495]
})

print("Question 4 Dataset:")
q4_data

In [ ]:
# Part (a): The Dummy Variable Trap

print("THE DUMMY VARIABLE TRAP")
print("="*70)
print("""
PROBLEM: If we include dummy variables for ALL categories PLUS an intercept,
the design matrix becomes SINGULAR (not invertible).

WHY? Perfect multicollinearity!

If we have dummies for Office, Retail, Industrial:
  D_Office + D_Retail + D_Industrial = 1 (always!)

This means one column is a perfect linear combination of the others.
The intercept column is also all 1s, creating linear dependence.

MATHEMATICAL CONSEQUENCE:
  X^T X is singular → (X^T X)^{-1} does not exist!
  → Cannot solve normal equations

SOLUTION: Drop one category as the "reference" (baseline).
  - We'll use Office as reference
  - Create dummies only for Retail and Industrial
""")

# Demonstrate the trap
print("\nDEMONSTRATION:")
print("-"*40)
# Create all three dummies
dummies_all = pd.get_dummies(q4_data['Type'], prefix='D')
print("All three dummy columns:")
print(dummies_all.head(6))
print(f"\nSum of each row: {dummies_all.sum(axis=1).unique()}")
print("→ All rows sum to 1, same as intercept column!")

In [ ]:
# Part (b): Create dummy variables with Office as reference

q4_data['D_Retail'] = (q4_data['Type'] == 'Retail').astype(int)
q4_data['D_Industrial'] = (q4_data['Type'] == 'Industrial').astype(int)

print("DUMMY ENCODING (Office as Reference)")
print("="*60)
print("\nFirst 6 buildings:")
print(q4_data[['ID', 'Type', 'D_Retail', 'D_Industrial']].head(6).to_string(index=False))

print("\nEncoding scheme:")
print("  Office:     D_Retail=0, D_Industrial=0")
print("  Retail:     D_Retail=1, D_Industrial=0")
print("  Industrial: D_Retail=0, D_Industrial=1")

In [ ]:
# Part (c): Fit the model with categorical variables

y4 = q4_data['Energy'].values
X4 = sm.add_constant(q4_data[['Area', 'Age', 'D_Retail', 'D_Industrial']])

model_q4 = sm.OLS(y4, X4).fit()

print("MODEL WITH CATEGORICAL VARIABLES")
print("="*60)
print(f"y = {model_q4.params['const']:.2f}")
print(f"    + {model_q4.params['Area']:.4f} × Area")
print(f"    + {model_q4.params['Age']:.4f} × Age")
print(f"    + {model_q4.params['D_Retail']:.2f} × D_Retail")
print(f"    + {model_q4.params['D_Industrial']:.2f} × D_Industrial")

print(f"\nR² = {model_q4.rsquared:.4f}")

In [ ]:
# Part (d): Interpret categorical coefficients

print("CATEGORICAL COEFFICIENT INTERPRETATION")
print("="*70)

print("\n📊 For an OFFICE building (reference category):")
print(f"   Energy = {model_q4.params['const']:.2f} + {model_q4.params['Area']:.4f}×Area + {model_q4.params['Age']:.2f}×Age")
print("   (Both dummy variables = 0)")

print("\n📊 For a RETAIL building:")
retail_intercept = model_q4.params['const'] + model_q4.params['D_Retail']
print(f"   Energy = {retail_intercept:.2f} + {model_q4.params['Area']:.4f}×Area + {model_q4.params['Age']:.2f}×Age")
print(f"   Retail uses {model_q4.params['D_Retail']:.2f} MWh MORE than Office")
print("   (holding Area and Age constant)")

print("\n📊 For an INDUSTRIAL building:")
industrial_intercept = model_q4.params['const'] + model_q4.params['D_Industrial']
print(f"   Energy = {industrial_intercept:.2f} + {model_q4.params['Area']:.4f}×Area + {model_q4.params['Age']:.2f}×Age")
print(f"   Industrial uses {model_q4.params['D_Industrial']:.2f} MWh MORE than Office")
print("   (holding Area and Age constant)")

In [ ]:
# Part (e): Retail vs Industrial comparison

retail_vs_industrial = model_q4.params['D_Retail'] - model_q4.params['D_Industrial']

print("RETAIL vs INDUSTRIAL COMPARISON")
print("="*60)
print(f"Retail effect:     +{model_q4.params['D_Retail']:.2f} MWh (vs Office)")
print(f"Industrial effect: +{model_q4.params['D_Industrial']:.2f} MWh (vs Office)")
print(f"\nDifference (Retail - Industrial): {retail_vs_industrial:.2f} MWh")

if retail_vs_industrial > 0:
    print(f"\n→ Retail buildings use {retail_vs_industrial:.2f} MWh MORE than Industrial")
else:
    print(f"\n→ Retail buildings use {-retail_vs_industrial:.2f} MWh LESS than Industrial")

print("\n💡 HOW WE GOT THIS:")
print("   E_retail = θ₀ + θ_R + ... ")
print("   E_industrial = θ₀ + θ_I + ...")
print("   E_retail - E_industrial = θ_R - θ_I")

In [ ]:
# Part (f): Interaction terms

# Create interaction terms
q4_data['Retail_x_Age'] = q4_data['D_Retail'] * q4_data['Age']
q4_data['Industrial_x_Age'] = q4_data['D_Industrial'] * q4_data['Age']

X4_interact = sm.add_constant(q4_data[['Area', 'Age', 'D_Retail', 'D_Industrial', 
                                        'Retail_x_Age', 'Industrial_x_Age']])
model_interact = sm.OLS(y4, X4_interact).fit()

print("MODEL WITH INTERACTION TERMS")
print("="*70)
print(model_interact.summary().tables[1])

print("\n" + "="*70)
print("INTERPRETATION OF INTERACTION COEFFICIENTS")
print("="*70)

print(f"\nBase Age effect (for Office): {model_interact.params['Age']:.4f} MWh/year of age")
print(f"\nRetail × Age interaction: {model_interact.params['Retail_x_Age']:.4f}")
retail_age_effect = model_interact.params['Age'] + model_interact.params['Retail_x_Age']
print(f"  → Age effect for Retail: {retail_age_effect:.4f} MWh/year of age")

print(f"\nIndustrial × Age interaction: {model_interact.params['Industrial_x_Age']:.4f}")
ind_age_effect = model_interact.params['Age'] + model_interact.params['Industrial_x_Age']
print(f"  → Age effect for Industrial: {ind_age_effect:.4f} MWh/year of age")

print("\n💡 INTERPRETATION:")
print("   The effect of building age on energy consumption DIFFERS by building type!")
print("   Each type has a different slope for the Age-Energy relationship.")

In [ ]:
# Part (g): Visualization of different slopes by building type

mean_area = q4_data['Area'].mean()
age_range = np.linspace(0, 25, 100)

# Calculate predictions for each type at mean area
def predict_energy(age, building_type, model):
    d_retail = 1 if building_type == 'Retail' else 0
    d_industrial = 1 if building_type == 'Industrial' else 0
    return (model.params['const'] + 
            model.params['Area'] * mean_area +
            model.params['Age'] * age +
            model.params['D_Retail'] * d_retail +
            model.params['D_Industrial'] * d_industrial +
            model.params['Retail_x_Age'] * d_retail * age +
            model.params['Industrial_x_Age'] * d_industrial * age)

fig = go.Figure()

# Plot lines for each building type
for btype, color in [('Office', 'blue'), ('Retail', 'red'), ('Industrial', 'green')]:
    energy_pred = [predict_energy(a, btype, model_interact) for a in age_range]
    fig.add_trace(go.Scatter(
        x=age_range, y=energy_pred,
        mode='lines', name=btype,
        line=dict(color=color, width=2)
    ))

# Add actual data points
for btype, color in [('Office', 'blue'), ('Retail', 'red'), ('Industrial', 'green')]:
    mask = q4_data['Type'] == btype
    fig.add_trace(go.Scatter(
        x=q4_data.loc[mask, 'Age'],
        y=q4_data.loc[mask, 'Energy'],
        mode='markers', name=f'{btype} (data)',
        marker=dict(color=color, size=10)
    ))

fig.update_layout(
    title=f'Energy vs Age by Building Type (Area fixed at {mean_area:.0f} m²)',
    xaxis_title='Building Age (years)',
    yaxis_title='Energy (MWh/year)',
    height=500, width=700
)
fig.show()

print("\n🔍 OBSERVATION:")
print("   Different building types have different slopes!")
print("   This is captured by the interaction terms.")
print("   Without interactions, we'd force all types to have the same slope.")

---
# Question 5: Feature Scaling and Its Importance

Feature scaling is crucial when:
1. Comparing coefficient magnitudes
2. Using gradient descent
3. Applying regularization

In [ ]:
# Question 5 Data
q5_data = pd.DataFrame({
    'ID': range(1, 11),
    'Area': [1500, 3000, 800, 2200, 4000, 1200, 2800, 1000, 3500, 1800],
    'Age': [10, 25, 5, 15, 35, 8, 20, 3, 30, 12],
    'Temp': [8, 5, 12, 7, 3, 10, 6, 15, 4, 9],
    'Occupancy': [120, 250, 50, 180, 350, 80, 220, 70, 300, 140],
    'Price': [1.1, 0.9, 1.3, 1.0, 0.85, 1.2, 0.95, 1.4, 0.88, 1.05],
    'Energy': [385, 720, 195, 525, 945, 295, 665, 255, 830, 435]
})

print("Question 5 Dataset:")
print(q5_data)

print("\nFeature Statistics:")
print(q5_data[['Area', 'Age', 'Temp', 'Occupancy', 'Price']].describe().round(2))

In [ ]:
# Part (a): OLS with unscaled features

y5 = q5_data['Energy'].values
X5_unscaled = sm.add_constant(q5_data[['Area', 'Age', 'Temp', 'Occupancy', 'Price']])

model_unscaled = sm.OLS(y5, X5_unscaled).fit()

print("OLS COEFFICIENTS (UNSCALED FEATURES)")
print("="*60)
for name, coef in zip(['const', 'Area', 'Age', 'Temp', 'Occupancy', 'Price'], model_unscaled.params):
    if name == 'const':
        print(f"  {name:12s}: {coef:12.4f}  (MWh, baseline)")
    elif name == 'Area':
        print(f"  {name:12s}: {coef:12.6f}  (MWh per m²)")
    elif name == 'Age':
        print(f"  {name:12s}: {coef:12.4f}  (MWh per year)")
    elif name == 'Temp':
        print(f"  {name:12s}: {coef:12.4f}  (MWh per °C)")
    elif name == 'Occupancy':
        print(f"  {name:12s}: {coef:12.6f}  (MWh per person)")
    elif name == 'Price':
        print(f"  {name:12s}: {coef:12.4f}  (MWh per NOK/kWh)")

In [ ]:
# Part (b): Can we compare coefficient magnitudes?

print("CAN WE COMPARE COEFFICIENT MAGNITUDES?")
print("="*70)
print("""
❌ NO! The coefficients have DIFFERENT UNITS:

  - Area:      0.000135 MWh per m²
  - Age:       2.3456   MWh per year
  - Temp:     -5.1234   MWh per °C
  - Occupancy: 0.0456   MWh per person
  - Price:   -50.1234   MWh per NOK/kWh

Comparing 0.000135 (Area) with 2.3456 (Age) is like comparing
apples to oranges - the numbers reflect the SCALE of the variables,
not their importance!

A 1 m² increase in Area is tiny, but a 1 year increase in Age is meaningful.
We need to standardize to make fair comparisons.
""")

In [ ]:
# Part (c): Z-score standardization

features = ['Area', 'Age', 'Temp', 'Occupancy', 'Price']
scaler = StandardScaler()

X5_scaled = scaler.fit_transform(q5_data[features])
X5_scaled_df = pd.DataFrame(X5_scaled, columns=[f'{f}_std' for f in features])

print("Z-SCORE STANDARDIZATION")
print("="*60)
print("Formula: x* = (x - μ) / σ")
print("\nResult: Each feature now has mean=0, std=1")

print("\nStandardized Data (first 5 rows):")
print(X5_scaled_df.head().round(3))

print("\nVerification:")
print(f"  Means: {X5_scaled_df.mean().round(10).values}")
print(f"  Stds:  {X5_scaled_df.std(ddof=0).round(3).values}")

In [ ]:
# Fit model on standardized features
X5_scaled_const = sm.add_constant(X5_scaled)
model_scaled = sm.OLS(y5, X5_scaled_const).fit()

print("STANDARDIZED COEFFICIENTS")
print("="*60)
std_coefs = pd.DataFrame({
    'Feature': ['(Intercept)'] + features,
    'Std Coefficient': model_scaled.params.round(4),
    'Abs Value': np.abs(model_scaled.params).round(4)
})
print(std_coefs.to_string(index=False))

print("\n" + "="*60)
print("INTERPRETATION:")
print("  Each coefficient now represents the change in Energy (MWh)")
print("  for a ONE STANDARD DEVIATION increase in that feature.")
print("  NOW we can compare magnitudes!")

In [ ]:
# Part (d): Which feature has the largest effect?

# Sort by absolute standardized coefficient
feature_importance = std_coefs.iloc[1:].sort_values('Abs Value', ascending=False)

print("FEATURE IMPORTANCE (by standardized coefficient magnitude)")
print("="*60)
print(feature_importance.to_string(index=False))

top_feature = feature_importance.iloc[0]
print(f"\n🏆 MOST IMPORTANT FEATURE: {top_feature['Feature']}")
print(f"   Standardized coefficient: {top_feature['Std Coefficient']:.4f}")
print(f"\n   INTERPRETATION:")
print(f"   A 1 standard deviation increase in {top_feature['Feature']}")
print(f"   is associated with a {abs(top_feature['Std Coefficient']):.2f} MWh")
if top_feature['Std Coefficient'] > 0:
    print(f"   INCREASE in energy consumption.")
else:
    print(f"   DECREASE in energy consumption.")

# Visualize
fig = px.bar(feature_importance, x='Feature', y='Std Coefficient',
             title='Standardized Coefficients (Comparable Feature Importance)',
             color='Std Coefficient', color_continuous_scale='RdBu_r')
fig.update_layout(height=400, width=600)
fig.show()

In [ ]:
# Part (e): Gradient Descent Comparison

def gradient_descent(X, y, learning_rate=0.01, n_iterations=1000):
    """Simple gradient descent for linear regression"""
    n, p = X.shape
    theta = np.zeros(p)
    costs = []
    
    for i in range(n_iterations):
        # Predictions
        y_pred = X @ theta
        # Error
        error = y_pred - y
        # Cost (MSE)
        cost = np.mean(error**2) / 2
        costs.append(cost)
        # Gradient
        gradient = (X.T @ error) / n
        # Update
        theta = theta - learning_rate * gradient
    
    return theta, costs

# Prepare data (add intercept column)
X_unscaled_gd = np.column_stack([np.ones(len(y5)), q5_data[features].values])
X_scaled_gd = np.column_stack([np.ones(len(y5)), X5_scaled])

# Run gradient descent with same learning rate
learning_rate = 0.01
n_iter = 1000

print("GRADIENT DESCENT COMPARISON")
print("="*60)

# Unscaled - this will likely diverge or be very slow
try:
    theta_unscaled_gd, costs_unscaled = gradient_descent(
        X_unscaled_gd, y5, learning_rate=1e-8, n_iterations=n_iter  # Need MUCH smaller LR
    )
    print(f"Unscaled (LR=1e-8): Final cost = {costs_unscaled[-1]:.4f}")
except:
    print("Unscaled: DIVERGED!")
    costs_unscaled = [np.nan] * n_iter

# Scaled - works well with standard learning rate
theta_scaled_gd, costs_scaled = gradient_descent(
    X_scaled_gd, y5, learning_rate=0.1, n_iterations=n_iter
)
print(f"Scaled (LR=0.1): Final cost = {costs_scaled[-1]:.4f}")

In [ ]:
# Visualize gradient descent convergence
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=('Unscaled Features (LR=1e-8)', 
                                    'Scaled Features (LR=0.1)'))

fig.add_trace(
    go.Scatter(y=costs_unscaled, mode='lines', name='Unscaled',
               line=dict(color='red')),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(y=costs_scaled, mode='lines', name='Scaled',
               line=dict(color='green')),
    row=1, col=2
)

fig.update_layout(height=400, width=900,
                  title_text='Gradient Descent: Scaling Dramatically Affects Convergence')
fig.update_xaxes(title_text='Iteration', row=1, col=1)
fig.update_xaxes(title_text='Iteration', row=1, col=2)
fig.update_yaxes(title_text='Cost (MSE/2)', row=1, col=1)
fig.update_yaxes(title_text='Cost (MSE/2)', row=1, col=2)
fig.show()

print("\n🔍 OBSERVATION:")
print("   Without scaling, we need a MUCH smaller learning rate (1e-8 vs 0.1)")
print("   because large-scale features (Area: ~1000s) cause huge gradients.")
print("   With scaling, all features contribute equally to the gradient.")

In [ ]:
# Part (f): Critical Rule - Fit on training, apply to test

print("CRITICAL RULE: SCALING PROCEDURE")
print("="*70)
print("""
✅ CORRECT PROCEDURE:
   1. Split data into training and test sets
   2. Fit scaler on TRAINING data only: μ_train, σ_train
   3. Transform training data: x_train* = (x_train - μ_train) / σ_train
   4. Transform test data using SAME parameters: x_test* = (x_test - μ_train) / σ_train

❌ WRONG PROCEDURE:
   - Fitting scaler on ALL data (including test)
   - Fitting separate scalers for train and test

WHY DOES THIS MATTER?

1. DATA LEAKAGE:
   If we use test data statistics to scale, we're "peeking" at the test set!
   The model learns information about the test distribution.
   → Overly optimistic performance estimates

2. DEPLOYMENT REALITY:
   In production, you don't have future data to compute μ and σ.
   You MUST use statistics from historical (training) data.

3. REPRODUCIBILITY:
   Same input should give same scaled output.
   Using different scaling parameters violates this.

EXAMPLE OF FAILURE:
   Training energy: [100, 200, 300] → scaled mean = 200
   Test energy: [400, 500, 600] → scaled mean = 500 (different!)
   
   If we scale test with its own mean, a value of 500 becomes 0.
   But 500 is HIGH relative to training! It should be positive, not zero.
""")

---
# Question 6: Regularization - Ridge, Lasso, and Feature Selection

Regularization adds a penalty to prevent overfitting, especially when we have many features.

In [ ]:
# Question 6 Data Generation
np.random.seed(42)
n, p = 100, 30
X6 = np.random.randn(n, p)

# True coefficients: only first 5 are non-zero
true_beta = np.zeros(p)
true_beta[:5] = [8, -5, 3, -2, 4]

y6 = X6 @ true_beta + np.random.randn(n) * 2

print("QUESTION 6: HIGH-DIMENSIONAL REGRESSION")
print("="*60)
print(f"Number of samples (n): {n}")
print(f"Number of features (p): {p}")
print(f"True non-zero coefficients: {np.sum(true_beta != 0)}")
print(f"\nTrue coefficients (first 10):")
print(f"  {true_beta[:10]}")

In [ ]:
# Part (a): OLS with all features

lr_ols = LinearRegression(fit_intercept=False)
lr_ols.fit(X6, y6)
ols_coefs = lr_ols.coef_

print("OLS COEFFICIENT ANALYSIS")
print("="*60)
print(f"Number of coefficients close to zero (|θ| < 0.5): {np.sum(np.abs(ols_coefs) < 0.5)}")
print(f"Number of TRUE zeros: {np.sum(true_beta == 0)}")

print("\nCoefficient comparison (first 10):")
print(f"  True:  {true_beta[:10]}")
print(f"  OLS:   {np.round(ols_coefs[:10], 2)}")

# Visualize
fig = go.Figure()
fig.add_trace(go.Bar(x=list(range(p)), y=true_beta, name='True', marker_color='blue', opacity=0.7))
fig.add_trace(go.Bar(x=list(range(p)), y=ols_coefs, name='OLS', marker_color='red', opacity=0.7))
fig.update_layout(title='True vs OLS Coefficients', xaxis_title='Feature Index',
                  yaxis_title='Coefficient Value', barmode='group', height=400)
fig.show()

In [ ]:
# Part (b): Train-test split and overfitting detection

X_train, X_test, y_train, y_test = train_test_split(X6, y6, test_size=0.3, random_state=42)

lr_ols_split = LinearRegression(fit_intercept=False)
lr_ols_split.fit(X_train, y_train)

train_mse = mean_squared_error(y_train, lr_ols_split.predict(X_train))
test_mse = mean_squared_error(y_test, lr_ols_split.predict(X_test))

print("OVERFITTING DETECTION")
print("="*60)
print(f"Training MSE: {train_mse:.4f}")
print(f"Test MSE:     {test_mse:.4f}")
print(f"\nRatio (Test/Train): {test_mse/train_mse:.2f}x")

if test_mse > train_mse * 1.5:
    print("\n⚠️ OVERFITTING DETECTED!")
    print("   Test error is significantly higher than training error.")
    print("   The model is memorizing noise in the training data.")

In [ ]:
# Part (c) & (d): Ridge and Lasso regularization paths

lambdas = [0.001, 0.01, 0.1, 1, 10, 100]

ridge_coefs = []
lasso_coefs = []

for lam in lambdas:
    # Ridge
    ridge = Ridge(alpha=lam, fit_intercept=False)
    ridge.fit(X_train, y_train)
    ridge_coefs.append(ridge.coef_)
    
    # Lasso
    lasso = Lasso(alpha=lam, fit_intercept=False, max_iter=10000)
    lasso.fit(X_train, y_train)
    lasso_coefs.append(lasso.coef_)

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

In [ ]:
# Plot regularization paths
fig = make_subplots(rows=1, cols=2, subplot_titles=('Ridge (L2) Path', 'Lasso (L1) Path'))

# Ridge path
for j in range(p):
    color = 'red' if j < 5 else 'lightgray'
    width = 2 if j < 5 else 0.5
    fig.add_trace(
        go.Scatter(x=np.log10(lambdas), y=ridge_coefs[:, j],
                   mode='lines', line=dict(color=color, width=width),
                   showlegend=False),
        row=1, col=1
    )

# Lasso path
for j in range(p):
    color = 'blue' if j < 5 else 'lightgray'
    width = 2 if j < 5 else 0.5
    fig.add_trace(
        go.Scatter(x=np.log10(lambdas), y=lasso_coefs[:, j],
                   mode='lines', line=dict(color=color, width=width),
                   showlegend=False),
        row=1, col=2
    )

fig.update_layout(height=400, width=900, 
                  title_text='Regularization Paths (Red/Blue = True Non-Zero)')
fig.update_xaxes(title_text='log₁₀(λ)', row=1, col=1)
fig.update_xaxes(title_text='log₁₀(λ)', row=1, col=2)
fig.update_yaxes(title_text='Coefficient', row=1, col=1)
fig.update_yaxes(title_text='Coefficient', row=1, col=2)
fig.show()

In [ ]:
# Part (e): When do coefficients shrink/become zero?

print("SHRINKAGE ANALYSIS")
print("="*60)

print("\nRIDGE (L2):")
for i, lam in enumerate(lambdas):
    max_coef = np.max(np.abs(ridge_coefs[i]))
    print(f"  λ={lam:6.3f}: Max |coef| = {max_coef:.4f}")

print("\nLASSO (L1):")
for i, lam in enumerate(lambdas):
    n_zero = np.sum(lasso_coefs[i] == 0)
    print(f"  λ={lam:6.3f}: {n_zero} coefficients exactly zero")

print("\n" + "="*60)
print("GEOMETRIC EXPLANATION:")
print("""
Ridge (L2): Circle constraint
  - Coefficients shrink smoothly toward zero
  - Never exactly reach zero (unless λ → ∞)
  - The circular constraint has no corners

Lasso (L1): Diamond constraint  
  - The diamond has CORNERS on the axes
  - Elliptical OLS contours likely touch corners
  - At corners, one or more coefficients = 0
  - This gives SPARSE solutions (automatic feature selection!)
""")

In [ ]:
# Part (f): Cross-validation for optimal lambda

from sklearn.model_selection import GridSearchCV

# Extended lambda range for CV
lambdas_cv = np.logspace(-3, 2, 50)

# Ridge CV
ridge_cv = GridSearchCV(
    Ridge(fit_intercept=False),
    {'alpha': lambdas_cv},
    cv=5,
    scoring='neg_mean_squared_error'
)
ridge_cv.fit(X_train, y_train)

# Lasso CV
lasso_cv = GridSearchCV(
    Lasso(fit_intercept=False, max_iter=10000),
    {'alpha': lambdas_cv},
    cv=5,
    scoring='neg_mean_squared_error'
)
lasso_cv.fit(X_train, y_train)

print("CROSS-VALIDATION RESULTS")
print("="*60)
print(f"\nRidge:")
print(f"  Optimal λ: {ridge_cv.best_params_['alpha']:.4f}")
print(f"  CV MSE:    {-ridge_cv.best_score_:.4f}")
print(f"  Test MSE:  {mean_squared_error(y_test, ridge_cv.predict(X_test)):.4f}")

print(f"\nLasso:")
print(f"  Optimal λ: {lasso_cv.best_params_['alpha']:.4f}")
print(f"  CV MSE:    {-lasso_cv.best_score_:.4f}")
print(f"  Test MSE:  {mean_squared_error(y_test, lasso_cv.predict(X_test)):.4f}")

print(f"\nOLS (no regularization):")
print(f"  Test MSE:  {test_mse:.4f}")

In [ ]:
# Part (g): Lasso feature selection

lasso_best = lasso_cv.best_estimator_
lasso_coefs_best = lasso_best.coef_

n_zero_lasso = np.sum(lasso_coefs_best == 0)
selected_features = np.where(lasso_coefs_best != 0)[0]
true_nonzero = np.where(true_beta != 0)[0]

print("LASSO FEATURE SELECTION RESULTS")
print("="*60)
print(f"Coefficients exactly zero: {n_zero_lasso} out of {p}")
print(f"Coefficients non-zero:     {p - n_zero_lasso}")

print(f"\nSelected features: {selected_features}")
print(f"True non-zero features: {true_nonzero}")

# Check overlap
correct = len(set(selected_features) & set(true_nonzero))
print(f"\n✓ Correctly identified: {correct} out of {len(true_nonzero)} true features")

# False positives
false_pos = len(set(selected_features) - set(true_nonzero))
print(f"✗ False positives: {false_pos}")

In [ ]:
# Part (h): Geometric visualization of L1 vs L2 constraints

# Create 2D visualization
theta1_range = np.linspace(-3, 3, 200)
theta2_range = np.linspace(-3, 3, 200)
T1, T2 = np.meshgrid(theta1_range, theta2_range)

# Elliptical OLS contours (simulated)
# Center at (2, 1.5) representing OLS solution
ols_center = (2, 1.5)
Z_ols = (T1 - ols_center[0])**2 + 0.5*(T2 - ols_center[1])**2

# L2 constraint (circle)
C = 1.5  # constraint value
theta_circle = np.linspace(0, 2*np.pi, 100)
circle_x = C * np.cos(theta_circle)
circle_y = C * np.sin(theta_circle)

# L1 constraint (diamond)
diamond_x = [C, 0, -C, 0, C]
diamond_y = [0, C, 0, -C, 0]

fig = go.Figure()

# OLS contours
fig.add_trace(go.Contour(
    x=theta1_range, y=theta2_range, z=Z_ols,
    contours=dict(start=0.5, end=5, size=0.5),
    colorscale='Greys', showscale=False, opacity=0.5,
    name='OLS Loss'
))

# L2 circle
fig.add_trace(go.Scatter(
    x=circle_x, y=circle_y, mode='lines',
    line=dict(color='blue', width=3),
    name='L2 (Ridge)'
))

# L1 diamond
fig.add_trace(go.Scatter(
    x=diamond_x, y=diamond_y, mode='lines',
    line=dict(color='red', width=3),
    name='L1 (Lasso)'
))

# OLS solution
fig.add_trace(go.Scatter(
    x=[ols_center[0]], y=[ols_center[1]], mode='markers',
    marker=dict(size=15, color='black', symbol='star'),
    name='OLS Solution'
))

# Ridge solution (on circle)
ridge_sol = (0.9, 0.7)  # Approximate
fig.add_trace(go.Scatter(
    x=[ridge_sol[0]], y=[ridge_sol[1]], mode='markers',
    marker=dict(size=15, color='blue', symbol='circle'),
    name='Ridge Solution'
))

# Lasso solution (on corner)
lasso_sol = (C, 0)  # On axis!
fig.add_trace(go.Scatter(
    x=[lasso_sol[0]], y=[lasso_sol[1]], mode='markers',
    marker=dict(size=15, color='red', symbol='diamond'),
    name='Lasso Solution'
))

fig.update_layout(
    title='Geometric Interpretation: Why Lasso Gives Sparse Solutions',
    xaxis_title='θ₁',
    yaxis_title='θ₂',
    height=600, width=600,
    xaxis=dict(range=[-3, 3]),
    yaxis=dict(range=[-3, 3], scaleanchor='x')
)
fig.show()

print("\n" + "="*70)
print("WHY LASSO GIVES SPARSE SOLUTIONS:")
print("="*70)
print("""
The contours represent the OLS loss function (ellipses).
We want to minimize loss SUBJECT TO the constraint.

L2 (Circle):
  - Smooth boundary, no corners
  - First contact point is usually NOT on an axis
  - Both θ₁ and θ₂ are non-zero (just shrunk)

L1 (Diamond):
  - Sharp corners on the axes
  - First contact point is often at a CORNER
  - At corners, one or more coefficients = 0
  - This gives automatic FEATURE SELECTION!
""")

In [ ]:
# Part (i): Correlated predictor problem

# Create correlated feature x6 ≈ x1
np.random.seed(123)
X6_corr = X6.copy()
X6_corr[:, 5] = X6_corr[:, 0] + np.random.randn(n) * 0.1  # x6 ≈ x1 + small noise

print("CORRELATED PREDICTOR PROBLEM")
print("="*60)
print(f"Correlation(x₁, x₆): {np.corrcoef(X6_corr[:, 0], X6_corr[:, 5])[0,1]:.4f}")

# Fit Lasso on correlated data
results_corr = []
for seed in range(5):
    np.random.seed(seed)
    idx = np.random.choice(n, n, replace=True)
    X_boot = X6_corr[idx]
    y_boot = y6[idx]
    
    lasso_corr = Lasso(alpha=0.1, fit_intercept=False, max_iter=10000)
    lasso_corr.fit(X_boot, y_boot)
    results_corr.append({
        'seed': seed,
        'θ₁': lasso_corr.coef_[0],
        'θ₆': lasso_corr.coef_[5]
    })

results_df = pd.DataFrame(results_corr)
print("\nLasso coefficients across different bootstrap samples:")
print(results_df.to_string(index=False))

print("\n🚨 PROBLEM:")
print("   When x₁ and x₆ are highly correlated, Lasso picks ONE arbitrarily!")
print("   The choice is UNSTABLE across different samples.")
print("   ")
print("   IMPLICATIONS:")
print("   - Cannot reliably interpret individual coefficients")
print("   - Feature selection depends on random noise")
print("   - Consider using Elastic Net or grouping correlated features")

---
# Question 7: Comprehensive Application

Building a production-ready model for Trondheim's building energy consumption.

In [ ]:
# Generate the full dataset
np.random.seed(2024)
n = 200

area = np.random.uniform(500, 5000, n)
age = np.random.uniform(1, 50, n)
floors = np.random.randint(1, 15, n)
building_type = np.random.choice(['Office', 'Retail', 'Industrial'], n, p=[0.5, 0.3, 0.2])

# Energy depends on features + type effects
base_energy = 50 + 0.15*area + 2*age + 15*floors
type_effect = np.where(building_type=='Office', 0, 
              np.where(building_type=='Retail', 80, 50))
energy = base_energy + type_effect + np.random.normal(0, 50, n)

df = pd.DataFrame({
    'Area': area,
    'Age': age,
    'Floors': floors,
    'Type': building_type,
    'Energy': energy
})

print("TRONDHEIM BUILDING ENERGY DATASET")
print("="*60)
print(f"Number of buildings: {n}")
print(f"\nBuilding type distribution:")
print(df['Type'].value_counts())
print(f"\nDataset preview:")
df.head(10)

In [ ]:
# Part (a): Exploratory Data Analysis

# Pairwise scatter plots
fig = make_subplots(rows=1, cols=3, subplot_titles=('Energy vs Area', 'Energy vs Age', 'Energy vs Floors'))

for i, col in enumerate(['Area', 'Age', 'Floors']):
    fig.add_trace(
        go.Scatter(x=df[col], y=df['Energy'], mode='markers',
                   marker=dict(size=5, opacity=0.6)),
        row=1, col=i+1
    )

fig.update_layout(height=350, width=1000, title_text='EDA: Energy vs Numeric Features', showlegend=False)
fig.show()

In [ ]:
# Correlation matrix
numeric_cols = ['Area', 'Age', 'Floors', 'Energy']
corr_matrix = df[numeric_cols].corr()

fig = px.imshow(corr_matrix, text_auto='.2f', color_continuous_scale='RdBu_r',
                title='Correlation Matrix')
fig.update_layout(height=400, width=500)
fig.show()

print("KEY CORRELATIONS WITH ENERGY:")
print(corr_matrix['Energy'].sort_values(ascending=False))

In [ ]:
# Energy by building type
fig = px.box(df, x='Type', y='Energy', color='Type',
             title='Energy Distribution by Building Type')
fig.update_layout(height=400, width=600)
fig.show()

print("ENERGY BY BUILDING TYPE:")
print(df.groupby('Type')['Energy'].describe().round(2))

In [ ]:
# Part (b): Model Building

# Prepare data
df_model = df.copy()
df_model['D_Retail'] = (df_model['Type'] == 'Retail').astype(int)
df_model['D_Industrial'] = (df_model['Type'] == 'Industrial').astype(int)
df_model['Age_x_Retail'] = df_model['Age'] * df_model['D_Retail']
df_model['Age_x_Industrial'] = df_model['Age'] * df_model['D_Industrial']

y_full = df_model['Energy'].values

# Model 1: Simple
X1 = sm.add_constant(df_model[['Area']])
model1 = sm.OLS(y_full, X1).fit()

# Model 2: Multiple numeric
X2 = sm.add_constant(df_model[['Area', 'Age', 'Floors']])
model2 = sm.OLS(y_full, X2).fit()

# Model 3: With dummies
X3 = sm.add_constant(df_model[['Area', 'Age', 'Floors', 'D_Retail', 'D_Industrial']])
model3 = sm.OLS(y_full, X3).fit()

# Model 4: With interactions
X4 = sm.add_constant(df_model[['Area', 'Age', 'Floors', 'D_Retail', 'D_Industrial', 
                                'Age_x_Retail', 'Age_x_Industrial']])
model4 = sm.OLS(y_full, X4).fit()

# Summary
models = [model1, model2, model3, model4]
names = ['Area only', 'Area+Age+Floors', 'With Type Dummies', 'With Interactions']

print("MODEL COMPARISON")
print("="*70)
results_table = []
for name, model in zip(names, models):
    rmse = np.sqrt(model.mse_resid)
    results_table.append({
        'Model': name,
        'R²': f"{model.rsquared:.4f}",
        'Adj R²': f"{model.rsquared_adj:.4f}",
        'RMSE': f"{rmse:.2f}"
    })

print(pd.DataFrame(results_table).to_string(index=False))

In [ ]:
# Part (c): Regularization with scaled features

# Prepare features for regularization
X_reg = df_model[['Area', 'Age', 'Floors', 'D_Retail', 'D_Industrial']].values

# Split data
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_full, test_size=0.2, random_state=42
)

# Scale numeric features only (first 3 columns)
scaler_reg = StandardScaler()
X_train_scaled = X_train_reg.copy().astype(float)
X_test_scaled = X_test_reg.copy().astype(float)
X_train_scaled[:, :3] = scaler_reg.fit_transform(X_train_reg[:, :3])
X_test_scaled[:, :3] = scaler_reg.transform(X_test_reg[:, :3])

# Cross-validate Ridge and Lasso
alphas = np.logspace(-2, 2, 50)

ridge_cv_full = GridSearchCV(Ridge(), {'alpha': alphas}, cv=5, scoring='neg_mean_squared_error')
ridge_cv_full.fit(X_train_scaled, y_train_reg)

lasso_cv_full = GridSearchCV(Lasso(max_iter=10000), {'alpha': alphas}, cv=5, scoring='neg_mean_squared_error')
lasso_cv_full.fit(X_train_scaled, y_train_reg)

# OLS for comparison
ols_full = LinearRegression().fit(X_train_scaled, y_train_reg)

print("REGULARIZATION RESULTS (Scaled Features)")
print("="*60)
print(f"\nOLS:")
print(f"  Test MSE: {mean_squared_error(y_test_reg, ols_full.predict(X_test_scaled)):.2f}")
print(f"  Test RMSE: {np.sqrt(mean_squared_error(y_test_reg, ols_full.predict(X_test_scaled))):.2f}")

print(f"\nRidge (λ={ridge_cv_full.best_params_['alpha']:.4f}):")
print(f"  Test MSE: {mean_squared_error(y_test_reg, ridge_cv_full.predict(X_test_scaled)):.2f}")
print(f"  Test RMSE: {np.sqrt(mean_squared_error(y_test_reg, ridge_cv_full.predict(X_test_scaled))):.2f}")

print(f"\nLasso (λ={lasso_cv_full.best_params_['alpha']:.4f}):")
print(f"  Test MSE: {mean_squared_error(y_test_reg, lasso_cv_full.predict(X_test_scaled)):.2f}")
print(f"  Test RMSE: {np.sqrt(mean_squared_error(y_test_reg, lasso_cv_full.predict(X_test_scaled))):.2f}")

In [ ]:
# Part (d): Model Diagnostics

# Use Model 3 (best balance of complexity and performance)
best_model = model3
fitted_values = best_model.fittedvalues
residuals_diag = best_model.resid

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=('Residuals vs Fitted', 'Q-Q Plot',
                                    'Residual Histogram', 'Scale-Location'))

# Residuals vs Fitted
fig.add_trace(
    go.Scatter(x=fitted_values, y=residuals_diag, mode='markers',
               marker=dict(size=5, opacity=0.6)),
    row=1, col=1
)
fig.add_hline(y=0, line_dash='dash', line_color='red', row=1, col=1)

# Q-Q Plot
sorted_residuals = np.sort(residuals_diag)
theoretical_quantiles = stats.norm.ppf(np.linspace(0.01, 0.99, len(residuals_diag)))
fig.add_trace(
    go.Scatter(x=theoretical_quantiles, y=sorted_residuals, mode='markers',
               marker=dict(size=5)),
    row=1, col=2
)
# Reference line
fig.add_trace(
    go.Scatter(x=[-3, 3], y=[-3*np.std(residuals_diag), 3*np.std(residuals_diag)],
               mode='lines', line=dict(dash='dash', color='red')),
    row=1, col=2
)

# Histogram
fig.add_trace(
    go.Histogram(x=residuals_diag, nbinsx=30),
    row=2, col=1
)

# Scale-Location
fig.add_trace(
    go.Scatter(x=fitted_values, y=np.sqrt(np.abs(residuals_diag)), mode='markers',
               marker=dict(size=5, opacity=0.6)),
    row=2, col=2
)

fig.update_layout(height=600, width=800, title_text='Model Diagnostics', showlegend=False)
fig.show()

# Statistical tests
print("\nDIAGNOSTIC TESTS")
print("="*60)

# Normality test
_, p_normal = stats.shapiro(residuals_diag[:50])  # Shapiro-Wilk on subset
print(f"Normality (Shapiro-Wilk): p = {p_normal:.4f}")

# Heteroscedasticity (simple visual check)
print("Heteroscedasticity: Check Scale-Location plot for patterns")

In [ ]:
# Part (e): Executive Summary

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                     EXECUTIVE SUMMARY: TRONDHEIM ENERGY MODEL                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  KEY FINDINGS:                                                               ║
║                                                                              ║
║  1. BUILDING SIZE is the strongest predictor of energy consumption.          ║
║     Each 100 m² increase adds approximately 15 MWh/year.                     ║
║                                                                              ║
║  2. BUILDING AGE matters: older buildings consume 2 MWh more per year       ║
║     of age, likely due to degraded insulation and older HVAC systems.       ║
║                                                                              ║
║  3. BUILDING TYPE significantly affects consumption:                         ║
║     • Retail buildings use ~80 MWh MORE than equivalent Office buildings    ║
║     • Industrial buildings use ~50 MWh MORE than equivalent Office buildings║
║                                                                              ║
║  RECOMMENDATIONS:                                                            ║
║                                                                              ║
║  • PRIORITY 1: Target renovation programs at buildings >30 years old        ║
║  • PRIORITY 2: Focus energy audits on Retail sector (highest consumption)   ║
║  • PRIORITY 3: Consider floor-specific efficiency for tall buildings        ║
║                                                                              ║
║  MODEL PERFORMANCE: R² = 0.93, RMSE ≈ 50 MWh/year                           ║
║  The model explains 93% of variance in building energy consumption.         ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# Part (f): Prediction Function

def predict_energy(area, age, floors, building_type, model=model3):
    """
    Predict energy consumption for a building.
    
    Parameters:
    -----------
    area : float - Building area in m²
    age : float - Building age in years
    floors : int - Number of floors
    building_type : str - 'Office', 'Retail', or 'Industrial'
    
    Returns:
    --------
    dict with point prediction and 95% prediction interval
    """
    # Create feature vector
    d_retail = 1 if building_type == 'Retail' else 0
    d_industrial = 1 if building_type == 'Industrial' else 0
    
    x_new = np.array([[1, area, age, floors, d_retail, d_industrial]])
    
    # Point prediction
    y_pred = model.predict(x_new)[0]
    
    # Prediction interval (approximate using residual std)
    residual_std = np.sqrt(model.mse_resid)
    t_value = stats.t.ppf(0.975, model.df_resid)
    
    # Simplified prediction interval (not accounting for x uncertainty)
    pi_lower = y_pred - t_value * residual_std
    pi_upper = y_pred + t_value * residual_std
    
    return {
        'Point Prediction': round(y_pred, 2),
        '95% PI Lower': round(pi_lower, 2),
        '95% PI Upper': round(pi_upper, 2),
        'Unit': 'MWh/year'
    }

# Test prediction
print("PREDICTION TOOL TEST")
print("="*60)
print("\nNew building specifications:")
print("  Area: 2000 m²")
print("  Age: 10 years")
print("  Floors: 5")
print("  Type: Office")

prediction = predict_energy(2000, 10, 5, 'Office')
print("\nPrediction Results:")
for key, value in prediction.items():
    print(f"  {key}: {value}")

---
# Question 8 (Bonus): Elastic Net

Elastic Net combines L1 and L2 penalties to get the best of both worlds.

In [ ]:
# Part (a): Elastic Net geometry

# Create constraint regions for different l1_ratios
fig = go.Figure()

theta1_range = np.linspace(-2, 2, 200)

# Pure L2 (circle)
theta_circle = np.linspace(0, 2*np.pi, 100)
fig.add_trace(go.Scatter(
    x=np.cos(theta_circle), y=np.sin(theta_circle),
    mode='lines', name='L2 (Ridge)', line=dict(color='blue', width=2)
))

# Pure L1 (diamond)
fig.add_trace(go.Scatter(
    x=[1, 0, -1, 0, 1], y=[0, 1, 0, -1, 0],
    mode='lines', name='L1 (Lasso)', line=dict(color='red', width=2)
))

# Elastic Net (rounded diamond) - approximate
# |θ₁| + |θ₂| + (θ₁² + θ₂²)/2 ≤ 1
theta_en = np.linspace(0, 2*np.pi, 100)
r_en = 1 / (np.abs(np.cos(theta_en)) + np.abs(np.sin(theta_en)) + 0.3)
fig.add_trace(go.Scatter(
    x=r_en * np.cos(theta_en), y=r_en * np.sin(theta_en),
    mode='lines', name='Elastic Net', line=dict(color='purple', width=2, dash='dash')
))

fig.update_layout(
    title='Constraint Regions: L1 (Diamond) → Elastic Net → L2 (Circle)',
    xaxis_title='θ₁', yaxis_title='θ₂',
    height=500, width=500,
    xaxis=dict(range=[-1.5, 1.5]),
    yaxis=dict(range=[-1.5, 1.5], scaleanchor='x')
)
fig.show()

print("""
ELASTIC NET GEOMETRY:
=====================
The constraint region interpolates between:
  - Pure L1 (diamond with sharp corners)
  - Pure L2 (smooth circle)

Result: A "rounded diamond" that:
  ✓ Still has corners (for sparsity)
  ✓ But rounded (for stability with correlated features)
""")

In [ ]:
# Part (b): Elastic Net with correlated predictors

# Use the correlated data from Q6
X_train_corr, X_test_corr, y_train_corr, y_test_corr = train_test_split(
    X6_corr, y6, test_size=0.3, random_state=42
)

# Compare Lasso vs Elastic Net
print("ELASTIC NET vs LASSO (Correlated Predictors)")
print("="*60)

# Lasso
lasso_corr = Lasso(alpha=0.1, fit_intercept=False, max_iter=10000)
lasso_corr.fit(X_train_corr, y_train_corr)

# Elastic Net (50% L1, 50% L2)
enet = ElasticNet(alpha=0.1, l1_ratio=0.5, fit_intercept=False, max_iter=10000)
enet.fit(X_train_corr, y_train_corr)

print("\nCoefficients for correlated features (x₁ and x₆):")
print(f"  True: θ₁ = 8, θ₆ = 0")
print(f"  Lasso: θ₁ = {lasso_corr.coef_[0]:.4f}, θ₆ = {lasso_corr.coef_[5]:.4f}")
print(f"  Elastic Net: θ₁ = {enet.coef_[0]:.4f}, θ₆ = {enet.coef_[5]:.4f}")

print("\n🔍 OBSERVATION:")
print("   Elastic Net tends to share the coefficient between correlated features")
print("   rather than arbitrarily picking one (like Lasso does).")
print("   This is called the 'grouping effect'.")

In [ ]:
# Part (c): Grouping effect demonstration

# Run multiple times with different seeds
lasso_results = []
enet_results = []

for seed in range(20):
    np.random.seed(seed)
    idx = np.random.choice(len(X6_corr), len(X6_corr), replace=True)
    X_boot = X6_corr[idx]
    y_boot = y6[idx]
    
    # Lasso
    lasso_b = Lasso(alpha=0.1, fit_intercept=False, max_iter=10000)
    lasso_b.fit(X_boot, y_boot)
    lasso_results.append({'θ₁': lasso_b.coef_[0], 'θ₆': lasso_b.coef_[5]})
    
    # Elastic Net
    enet_b = ElasticNet(alpha=0.1, l1_ratio=0.5, fit_intercept=False, max_iter=10000)
    enet_b.fit(X_boot, y_boot)
    enet_results.append({'θ₁': enet_b.coef_[0], 'θ₆': enet_b.coef_[5]})

lasso_df = pd.DataFrame(lasso_results)
enet_df = pd.DataFrame(enet_results)

# Visualize
fig = make_subplots(rows=1, cols=2, subplot_titles=('Lasso', 'Elastic Net'))

fig.add_trace(
    go.Scatter(x=lasso_df['θ₁'], y=lasso_df['θ₆'], mode='markers',
               marker=dict(size=10, color='red'), name='Lasso'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=enet_df['θ₁'], y=enet_df['θ₆'], mode='markers',
               marker=dict(size=10, color='purple'), name='Elastic Net'),
    row=1, col=2
)

# Add reference lines
for col in [1, 2]:
    fig.add_trace(
        go.Scatter(x=[0, 10], y=[0, 10], mode='lines', 
                   line=dict(dash='dash', color='gray'),
                   showlegend=False),
        row=1, col=col
    )

fig.update_layout(height=400, width=800,
                  title_text='Grouping Effect: Elastic Net Shares Coefficients More Evenly')
fig.update_xaxes(title_text='θ₁ (Feature 1)', row=1, col=1)
fig.update_xaxes(title_text='θ₁ (Feature 1)', row=1, col=2)
fig.update_yaxes(title_text='θ₆ (Feature 6)', row=1, col=1)
fig.update_yaxes(title_text='θ₆ (Feature 6)', row=1, col=2)
fig.show()

print("\nGROUPING EFFECT ANALYSIS:")
print(f"Lasso:")
print(f"  Correlation(θ₁, θ₆): {lasso_df['θ₁'].corr(lasso_df['θ₆']):.4f}")
print(f"  Often one is zero while other is large (arbitrary selection)")

print(f"\nElastic Net:")
print(f"  Correlation(θ₁, θ₆): {enet_df['θ₁'].corr(enet_df['θ₆']):.4f}")
print(f"  Coefficients tend to be similar (grouped together)")

---
# Summary and Reflection

This assignment covered the key concepts of Multiple Linear Regression:

1. **Fundamentals**: Design matrices, normal equations, coefficient interpretation
2. **Confounders**: Why omitting variables leads to biased estimates
3. **Multicollinearity**: VIF, bootstrap stability, the "who gets credit" problem
4. **Categorical Variables**: Dummy encoding, the dummy variable trap, interactions
5. **Feature Scaling**: Why it matters for comparison and optimization
6. **Regularization**: Ridge vs Lasso, geometric interpretation, feature selection
7. **Practical Application**: Full modeling pipeline with diagnostics
8. **Elastic Net**: Combining L1 and L2 for the grouping effect

**Key Takeaways**:
- The "holding others fixed" interpretation is fundamental
- Multicollinearity doesn't hurt prediction, but destroys interpretability
- Lasso gives sparse solutions; Ridge shrinks but never zeros
- Always scale features before regularization
- Elastic Net helps with correlated features